Inicio: Primera exploración de los datos de un solo mes (Enero del 2025).

- "Green taxis"

El archivo viene en formato parquet,.
Para hacer una primera exploración se utilizará duckdb, ya que es mucho más rápido.

In [1]:
# Instalación librerias
!pip install -q duckdb

# Cargar librerías
import os, requests, duckdb
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Conectar a la carpeta del drive
from google.colab import drive

drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/nyc-taxi"

Mounted at /content/drive


In [3]:
# Cargar los datos
con = duckdb.connect()
con.sql(f"CREATE VIEW trips AS SELECT * FROM read_parquet('{BASE}/green_tripdata_*.parquet')")
  #El * del nombre sirve para que cuando haya más meses a la carpeta los lea todos.
con.sql(f"CREATE VIEW zones AS SELECT * FROM read_csv_auto('{BASE}/taxi_zone_lookup.csv')")

In [7]:
#Cunatas filas existen y columnas

filas = con.sql ("SELECT count(*) FROM trips")
print(filas)

columnas = con.sql("describe trips")
print(columnas)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        48326 │
└──────────────┘

┌───────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name      │ column_type │  null   │   key   │ default │  extra  │
│        varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ VendorID              │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ lpep_pickup_datetime  │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ lpep_dropoff_datetime │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ store_and_fwd_flag    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ RatecodeID            │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ PULocationID          │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ DOLocationID          │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ passenger_count

No tiene las mismas variables que los taxis amarillos.
Hay que cambiar los nombres de las variables de fech ay hora para que cuadre con las otras observaciones (yellow taxi)


In [14]:
#mirar cuantos nulos tiene cada columna

def nulos_por_columna (tabla):
  cols = con.sql(f"DESCRIBE {tabla}").df()["column_name"].tolist()
  expr = ", ".join(f'count(*) FILTER (WHERE "{c}" IS NULL) AS "{c}"' for c in cols)
  return con.sql(f"SELECT count(*) AS filas, {expr} FROM {tabla}").df().T

nulos_por_columna("trips")

,0
filas,48326
VendorID,0
lpep_pickup_datetime,0
lpep_dropoff_datetime,0
store_and_fwd_flag,1836
RatecodeID,1836
PULocationID,0
DOLocationID,0
passenger_count,1836
trip_distance,0


columna ehail_fee va totalmente fuera, tiene todos los valores nulos.
Hay que mirar si las columnas con 1836 nulos son las mismas observaciones, para decidir si se eliminan o no.

In [19]:
#miarar si las columnas coinciden y así eliminar esas observaciones

nulos_observaciones = con.sql(""" SELECT (cbd_congestion_fee IS NULL) AS n1, (congestion_surcharge IS NULL) AS n2, (payment_type IS NULL) AS n3,
       (passenger_count IS NULL) AS n4, (RatecodeID IS NULL) AS n5,(store_and_fwd_flag IS NULL) AS n6,(trip_type IS NULL) AS n7,
       count(*) AS filas
FROM trips
GROUP BY ALL
ORDER BY filas DESC
""")
print(nulos_observaciones)

┌─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬───────┐
│   n1    │   n2    │   n3    │   n4    │   n5    │   n6    │   n7    │ filas │
│ boolean │ boolean │ boolean │ boolean │ boolean │ boolean │ boolean │ int64 │
├─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼───────┤
│ false   │ false   │ false   │ false   │ false   │ false   │ false   │ 46483 │
│ true    │ true    │ true    │ true    │ true    │ true    │ true    │  1836 │
│ false   │ false   │ false   │ false   │ false   │ false   │ true    │     7 │
└─────────┴─────────┴─────────┴─────────┴─────────┴─────────┴─────────┴───────┘



Eliminar columna ehail_fee y las 1836 obsarvaciones (un 3,8%). Hay 7 observaciones en trip_type que se quedaran como nulo, pero el resto de 1836 se eliminan.

#### Exploración datos

In [20]:
# payment_type
tipos_pagos = con.sql("select payment_type, count(*) as viajes from trips group by payment_type order by viajes DESC")
print(tipos_pagos)

# trip_type
tipos_viaje = con.sql("select trip_type, count(*) as viajes from trips group by trip_type order by viajes DESC")
print(tipos_viaje)

# cuantos viajes por zona de origen
origen = con.sql("""SELECT PULocationID, count(*) AS viajes FROM trips
                    group by PULocationID
                    order by viajes desc""")
print(origen)

# cuantos viajes por zona de destinación
destino = con.sql("""SELECT DOLocationID, count(*) AS viajes FROM trips
                      group by DOLocationID
                      order by viajes desc""")
print(destino)


# Cruce de pagos y origen
pagos_origen = con.sql("""SELECT PULocationID, payment_type, count(*) AS viajes FROM trips
                          group by PULocationID, payment_type
                          order by viajes desc""")
print(pagos_origen)

# Cruce de pagos y destinación
pagos_destino = con.sql("""SELECT DOLocationID, payment_type, count(*) AS viajes FROM trips
                            group by DOLocationID, payment_type
                            order by viajes desc""")
print(pagos_destino)

# Media de pago por zona de origen
media_origen = con.sql("""SELECT round(mean(Total_amount), 2) as pago_medio, PULocationID, count(*) AS viajes
                          from trips
                          group by PULocationID
                          order by pago_medio desc""")
print(media_origen)

# Media de pago por zona de destinación
media_destino = con.sql("""SELECT round(avg(Total_amount), 2) as pago_medio, DOLocationID, count(*) AS viajes
                            from trips
                            group by DOLocationID
                            order by pago_medio desc""")
print(media_destino)

# Descriptivos de distancia de los viajes
distancia = con.sql("""select count(*)  AS filas, count(trip_distance) AS no_nulos,
                        min(trip_distance) as minimo, max(trip_distance) as maximo,
                        round(avg(trip_distance), 2) as media, median(trip_distance) as mediana
                        from trips""")
print(distancia)

# Tiempo y distancia calculado por el aparato
fare = con.sql("""select count(*)  AS filas, count(fare_amount) AS no_nulos, min(fare_amount) as minimo,
                  max(fare_amount) as maximo, round(avg(fare_amount), 2) as media,
                  median(fare_amount) as mediana
                  from trips""")
print(fare)

#Pasajeros
pasajeros = con.sql("""select count(*)  AS filas, count(Passenger_count) AS no_nulos,
                        min(Passenger_count) as minimo, max(Passenger_count) as maximo,
                        round(avg(Passenger_count), 2) as media, median(Passenger_count) as mediana
                        from trips""")
print(pasajeros)

# Fechas
fechas = con.sql("""
SELECT strftime(lpep_pickup_datetime, '%Y-%m') AS mes, count(*) AS viajes
FROM trips GROUP BY mes ORDER BY mes
""")
print(fechas)

# Duración desplazamientos
duracion = con.sql("""
SELECT count(*) FILTER (WHERE lpep_dropoff_datetime <= lpep_pickup_datetime) AS duracion_cero_o_negativa,
       count(*) FILTER (WHERE date_diff('hour', lpep_pickup_datetime, lpep_dropoff_datetime) >= 5) AS mas_de_5_horas
FROM trips
""")
print(duracion)

# Duplicados
duplicados = con.sql("""
SELECT count(*) AS filas,
       count(DISTINCT (lpep_pickup_datetime, lpep_dropoff_datetime, PULocationID, DOLocationID, total_amount)) AS distintas
FROM trips
""")
print(duplicados)

┌──────────────┬────────┐
│ payment_type │ viajes │
│    int64     │ int64  │
├──────────────┼────────┤
│            1 │  34635 │
│            2 │  11424 │
│         NULL │   1836 │
│            3 │    332 │
│            4 │     98 │
│            5 │      1 │
└──────────────┴────────┘

┌───────────┬────────┐
│ trip_type │ viajes │
│   int64   │ int64  │
├───────────┼────────┤
│         1 │  44680 │
│      NULL │   1843 │
│         2 │   1803 │
└───────────┴────────┘

┌──────────────┬────────┐
│ PULocationID │ viajes │
│    int32     │ int64  │
├──────────────┼────────┤
│           74 │  11682 │
│           75 │   8065 │
│           43 │   2776 │
│           95 │   2242 │
│          166 │   2126 │
│           97 │   2004 │
│           41 │   1974 │
│           82 │   1804 │
│           65 │   1518 │
│          130 │   1325 │
│           ·  │      · │
│           ·  │      · │
│           ·  │      · │
│          175 │      1 │
│          178 │      1 │
│          184 │      1 │
│       

Que se puede observar:
- Distancia máxima: poner un limite ya que el máximo es de 84.700
- Fare negativo, mirar el q
- Duración del viaje igual o menor a 0 (48 casos)
- Duración del viaje más de 5 horas (241 casos)
- Mirar distribución de pasajeros y decidir el q


In [21]:
# Mirar que pasa con el fare negativo

cuantos_fare_hay = con.sql("""
SELECT count(*)                                                AS filas,
       count(*) FILTER (WHERE fare_amount < 0)                 AS fare_negativo,
       count(*) FILTER (WHERE fare_amount = 0)                 AS fare_cero,
       count(*) FILTER (WHERE total_amount < 0)                AS total_negativo,
       count(*) FILTER (WHERE fare_amount < 0 AND total_amount >= 0) AS fare_neg_total_pos,
       round(100.0 * count(*) FILTER (WHERE fare_amount < 0) / count(*), 2) AS pct_fare_negativo
FROM trips
""")

tipo_pago_fare = con.sql("""
SELECT payment_type,
       count(*) AS viajes,
       round(avg(fare_amount), 2)  AS fare_medio,
       round(avg(total_amount), 2) AS total_medio
FROM trips
WHERE fare_amount < 0
GROUP BY payment_type
ORDER BY viajes DESC
""")

ejemplo_fare = con.sql("""
SELECT fare_amount, extra, mta_tax, tip_amount, tolls_amount,
       improvement_surcharge, congestion_surcharge, total_amount,
       payment_type, trip_distance
FROM trips
WHERE fare_amount < 0
LIMIT 15
""")

print(cuantos_fare_hay)
print(tipo_pago_fare)
print(ejemplo_fare)

┌───────┬───────────────┬───────────┬────────────────┬────────────────────┬───────────────────┐
│ filas │ fare_negativo │ fare_cero │ total_negativo │ fare_neg_total_pos │ pct_fare_negativo │
│ int64 │     int64     │   int64   │     int64      │       int64        │      double       │
├───────┼───────────────┼───────────┼────────────────┼────────────────────┼───────────────────┤
│ 48326 │           150 │        41 │            153 │                  0 │              0.31 │
└───────┴───────────────┴───────────┴────────────────┴────────────────────┴───────────────────┘

┌──────────────┬────────┬────────────┬─────────────┐
│ payment_type │ viajes │ fare_medio │ total_medio │
│    int64     │ int64  │   double   │   double    │
├──────────────┼────────┼────────────┼─────────────┤
│            3 │    107 │      -9.63 │      -11.63 │
│            4 │     42 │     -13.44 │      -15.71 │
│            2 │      1 │       -3.7 │        -5.2 │
└──────────────┴────────┴────────────┴─────────────┘

In [22]:
# Distribución personas
# Distribución %
porcenatge_pasajeros = con.sql("""
SELECT passenger_count,
       count(*) AS viajes,
       round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
FROM trips
GROUP BY passenger_count
ORDER BY passenger_count
""")

# Valores raros de pasajeros son viajes adecuados
valores_raros_pasajeros = con.sql("""
SELECT CASE WHEN passenger_count IS NULL THEN 'nulo'
            WHEN passenger_count = 0     THEN '0'
            WHEN passenger_count <= 5    THEN '1 a 5'
            ELSE 'más de 5' END AS grupo,
       count(*) AS viajes,
       round(avg(fare_amount), 2)    AS fare_medio,
       round(median(trip_distance), 2) AS distancia_mediana
FROM trips
GROUP BY grupo
ORDER BY viajes DESC
""")

print(porcenatge_pasajeros)
print(valores_raros_pasajeros)

┌─────────────────┬────────┬────────┐
│ passenger_count │ viajes │  pct   │
│      int64      │ int64  │ double │
├─────────────────┼────────┼────────┤
│               0 │    647 │   1.34 │
│               1 │  39271 │  81.26 │
│               2 │   4196 │   8.68 │
│               3 │    381 │   0.79 │
│               4 │    171 │   0.35 │
│               5 │   1030 │   2.13 │
│               6 │    788 │   1.63 │
│               7 │      1 │    0.0 │
│               8 │      1 │    0.0 │
│               9 │      4 │   0.01 │
│            NULL │   1836 │    3.8 │
├─────────────────┴────────┴────────┤
│ 11 rows                 3 columns │
└───────────────────────────────────┘

┌──────────┬────────┬────────────┬───────────────────┐
│  grupo   │ viajes │ fare_medio │ distancia_mediana │
│ varchar  │ int64  │   double   │      double       │
├──────────┼────────┼────────────┼───────────────────┤
│ 1 a 5    │  45049 │      16.49 │              1.72 │
│ nulo     │   1836 │      24.46 │      

In [28]:
# Viajes con distancia alta

distancia_alta = con.sql("select trip_distance from trips order by trip_distance desc limit 22")
print(distancia_alta)

┌───────────────┐
│ trip_distance │
│    double     │
├───────────────┤
│      84731.57 │
│      81665.35 │
│      79524.59 │
│      68168.33 │
│      62123.18 │
│      41824.07 │
│      40034.73 │
│      39681.65 │
│      39013.94 │
│      37961.99 │
│      36666.53 │
│      35757.82 │
│      35027.73 │
│      34889.49 │
│      33841.89 │
│      32057.74 │
│      28478.48 │
│      28182.37 │
│      27589.71 │
│      26353.41 │
│       24017.2 │
│        122.26 │
├───────────────┤
│    22 rows    │
└───────────────┘



Para el fare_amount poner --> marcar is_adjustment = total_amount < 0 (153 filas, 0,3 %) y excluirlas de los análisis de demanda e ingresos. Igual que en taxis amarillos.

Para los pasajeros --> conservar todas las filas; tratar passenger_count = 0 y los valores de 7 en adelante como desconocido para cualquier análisis de pasajeros; no usar esta columna para decisiones importantes.

Para la distancia marcar que tienen que ser menores de 180 (esto también para los taxis amarillos)


Compara en igualdad de condiciones. Los verdes solo recogen en zonas exteriores y norte de Manhattan. Comparar su media con la de todos los amarillos mezcla dos mercados distintos. Filtra los amarillos a las mismas zonas de recogida que usan los verdes y verás si el servicio se comporta distinto (propinas, tarifa por minuto, forma de pago) o solo opera en otra geografía. Esa es la comparación que se puede defender en una entrevista. La pregunta de “taxi pedido por app frente a parado en la calle” se puede responder con trip_type: 1 es street-hail (lo paran en la calle) y 2 es dispatch (despacho por radio o app)

**Pregunta nueva:** ¿dónde compiten y dónde no? Es la idea que más valor da al proyecto. Calcula el porcentaje de verdes sobre el total de viajes de cada zona

**La carga de los datos tarda 1 segundo.**

**El resto se ejecutan al instante.**